In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, regularizers

np.random.seed(42)
tf.random.set_seed(42)
# Small dataset
X = np.array([1, 2, 3, 4, 5], dtype=float)
Y = np.array([3, 5, 7, 9, 11], dtype=float)

def gradient_descent(method, epochs=100, lr=0.01):

    w = 0.0
    b = 0.0
    n = len(X)

    for epoch in range(epochs):

        indices = np.random.permutation(n)

        if method == "Batch":
            batches = [indices]

        elif method == "Stochastic":
            batches = [[i] for i in indices]

        else:
            batches = [indices[i:i+2] for i in range(0, n, 2)]

        for batch in batches:

            xb = X[batch]
            yb = Y[batch]

            prediction = w * xb + b
            error = prediction - yb

            dw = 2 * np.mean(error * xb)
            db = 2 * np.mean(error)

            w = w - lr * dw
            b = b - lr * db

    mse = np.mean((w * X + b - Y) ** 2)

    return w, b, mse


for method in ["Batch", "Stochastic", "Mini-Batch"]:

    w, b, mse = gradient_descent(method)

    print("\nMethod:", method)
    print("Weight:", round(w, 4))
    print("Bias:", round(b, 4))
    print("MSE:", round(mse, 4))
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Use a subset
X_train = X_train[:12000]
y_train = y_train[:12000]

X_test = X_test[:3000]
y_test = y_test[:3000]

# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

# Validation dataset
X_val = X_train[-2000:]
y_val = y_train[-2000:]

X_train = X_train[:-2000]
y_train = y_train[:-2000]

# Flatten images
X_train = X_train.reshape(-1, 784)
X_val = X_val.reshape(-1, 784)
X_test = X_test.reshape(-1, 784)

print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)
print("Testing data:", X_test.shape)
class_names = [
    'T-shirt', 'Trouser', 'Pullover', 'Dress',
    'Coat', 'Sandal', 'Shirt', 'Sneaker',
    'Bag', 'Ankle Boot'
]

plt.figure(figsize=(10, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i].reshape(28, 28), cmap='gray')
    plt.title(class_names[y_train[i]])
    plt.axis('off')

plt.tight_layout()
plt.show()
def create_model():

    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    return model


optimizers = {
    'SGD Momentum': keras.optimizers.SGD(
        learning_rate=0.01, momentum=0.9
    ),

    'Nesterov SGD': keras.optimizers.SGD(
        learning_rate=0.01, momentum=0.9, nesterov=True
    ),

    'RMSprop': keras.optimizers.RMSprop(
        learning_rate=0.001
    ),

    'Adam': keras.optimizers.Adam(
        learning_rate=0.001
    )
}

optimizer_results = {}

for name, optimizer in optimizers.items():

    print("\nTraining:", name)

    model = create_model()

    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=128,
        verbose=1
    )

    optimizer_results[name] = history

    test_loss, test_acc = model.evaluate(
        X_test, y_test, verbose=0
    )

    print("Test Accuracy:", test_acc)
def train_learning_rate(schedule):

    model = create_model()

    if schedule == "Constant":

        lr = 0.001

    else:

        lr = keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=0.001,
            decay_steps=100,
            decay_rate=0.9
        )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=128,
        verbose=1
    )

    return history


constant_history = train_learning_rate("Constant")
scheduled_history = train_learning_rate("Scheduled")

plt.plot(constant_history.history['val_accuracy'],
         label='Constant LR')

plt.plot(scheduled_history.history['val_accuracy'],
         label='Scheduled LR')

plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.title("Constant vs Scheduled Learning Rate")
plt.legend()
plt.show()
def dropout_model(dropout_rate):

    model = keras.Sequential([
        layers.Input(shape=(784,)),

        layers.Dense(256, activation='relu'),

        layers.Dropout(dropout_rate),

        layers.Dense(128, activation='relu'),

        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


dropout_results = {}

for rate in [0.0, 0.3, 0.5]:

    print("\nDropout Rate:", rate)

    model = dropout_model(rate)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=128,
        verbose=1
    )

    dropout_results[rate] = history


for rate, history in dropout_results.items():

    plt.plot(
        history.history['val_accuracy'],
        label=f'Dropout {rate}'
    )

plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.title("Effect of Dropout")
plt.legend()
plt.show()
def l2_model():

    model = keras.Sequential([
        layers.Input(shape=(784,)),

        layers.Dense(
            256,
            activation='relu',
            kernel_regularizer=regularizers.l2(0.001)
        ),

        layers.Dense(
            128,
            activation='relu',
            kernel_regularizer=regularizers.l2(0.001)
        ),

        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


model = l2_model()

history_l2 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=128
)

print("L2 Test Accuracy:",
      model.evaluate(X_test, y_test, verbose=0)[1])
augmentation = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.1, 0.1)
])

def augmentation_model():

    model = keras.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Reshape((28, 28, 1)),
        augmentation,
        layers.Flatten(),

        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


X_train_images = X_train.reshape(-1, 28, 28)
X_val_images = X_val.reshape(-1, 28, 28)

model = augmentation_model()

history_aug = model.fit(
    X_train_images, y_train,
    validation_data=(X_val_images, y_val),
    epochs=10,
    batch_size=128
)

print("Augmentation Test Accuracy:",
      model.evaluate(X_test.reshape(-1, 28, 28),
                     y_test, verbose=0)[1])
model = create_model()

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history_early = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    callbacks=[early_stop]
)

print("Test Accuracy:",
      model.evaluate(X_test, y_test, verbose=0)[1])
def plain_mlp():

    inputs = keras.Input(shape=(784,))

    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dense(128, activation='relu')(x)

    outputs = layers.Dense(10, activation='softmax')(x)

    return keras.Model(inputs, outputs)


def residual_mlp():

    inputs = keras.Input(shape=(784,))

    shortcut = layers.Dense(256)(inputs)

    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dense(256)(x)

    x = layers.Add()([shortcut, x])
    x = layers.Activation('relu')(x)

    x = layers.Dense(128, activation='relu')(x)

    outputs = layers.Dense(10, activation='softmax')(x)

    return keras.Model(inputs, outputs)


models = {
    'Plain MLP': plain_mlp(),
    'Residual MLP': residual_mlp()
}

architecture_results = {}

for name, model in models.items():

    print("\nTraining:", name)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=128
    )

    architecture_results[name] = history

    loss, accuracy = model.evaluate(
        X_test, y_test, verbose=0
    )

    print(name, "Test Accuracy:", accuracy)


plt.figure(figsize=(8, 5))

for name, history in architecture_results.items():

    plt.plot(
        history.history['val_accuracy'],
        label=name
    )

plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy")
plt.title("Plain MLP vs Residual MLP")
plt.legend()
plt.show()
print("\nFINAL COMPARISON")

for name, history in optimizer_results.items():

    best_accuracy = max(history.history['val_accuracy'])

    print(name, ":", round(best_accuracy, 4))